In [34]:
import gensim.downloader as api
import ast
import torch
import torchvision.models as models
from torchvision.transforms import functional as F
from PIL import Image

import numpy as np

def get_roi_class_probabilities(image, boxes, model, preprocess):
    class_probs = []
    
    
    for box in boxes:
        
        temp_box = np.array(box)
        roi = image.crop(temp_box)
        
        # Preprocess the RoI
        roi_tensor = preprocess(roi).unsqueeze(0).to(device)
        
        with torch.no_grad():
            temp_probs = model(roi_tensor)
            probs = torch.nn.functional.softmax(temp_probs, dim=1)
            class_probs.append(probs)
    
    return torch.cat(class_probs)

def compute_weighted_embeddings(class_probs, embeddings_tensor):
    # Compute the weighted sum of embeddings using class probabilities
    weighted_embeddings = torch.mm(class_probs, embeddings_tensor)
    return weighted_embeddings

In [43]:

def get_semantic_embeddings(ml_elements, gif, feat, hand_indices):
    """
    ml_elements['clip']         : clip model (modified by ViPLO) to be used for extracting features
    ml_elements['clip_preproc'] : pre-processing function to be used to pre-process the features
    video                       : Object having the video
    feat                        : The current extracted feats also having the tracked bounding boxes
    """

    num_obj = int(feat['num_obj'])
    
    bbox_tensors = feat['bboxes'][:num_obj, :]
    
    obj_indices = []
    for i in range(num_obj):
        if i not in hand_indices:
            obj_indices.append(i)
    
    obj_bboxes = bbox_tensors[obj_indices, 5, :].tolist()
    
    frame = gif[5]
    
    roi_det_model = ml_elements['ROI_DET_MODEL']
    roi_preprocess = ml_elements['ROI_PREPROCESS']
    embeddings_tensor = ml_elements['imagenet_word2vec_embed']
    
    temp_class_probs = get_roi_class_probabilities(frame, obj_bboxes,
                                roi_det_model, roi_preprocess)
    
    temp_semantic_embeddings = compute_weighted_embeddings(temp_class_probs,
                                                           embeddings_tensor)
    
    semantic_embeddings = torch.zeros((12, 300), device=temp_semantic_embeddings.device)
    
    semantic_embeddings[obj_indices, :] = temp_semantic_embeddings
    semantic_embeddings[hand_indices, :] = ml_elements['hand_embedding']
    
    feat['semantic_embeddings'] = semantic_embeddings
    
    return feat

In [28]:
# Create all the various ml_elements

ml_elements = {}
device = torch.device('cuda:0')
ml_elements['device'] = device

import clip
clip_model, clip_preproc = clip.load("ViT-B/32")
clip_model = clip_model.eval().to(ml_elements['device'])

ml_elements['VIPLO_CLIP']  = clip_model
ml_elements['VIPLO_PRE_PROC'] = clip_preproc


import sys
sys.path.append("/workspace/work/ral_revise_and_resubmit/segment-anything")

from segment_anything import sam_model_registry, SamPredictor

sam_checkpoint = "/workspace/work/ral_revise_and_resubmit/segment-anything/sam_vit_h_4b8939.pth"
model_type = "vit_h"

device = "cuda:1"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

predictor = SamPredictor(sam)
ml_elements['SAM_predictor'] = predictor




import torch
import torchvision.models as models
from torchvision.transforms import functional as F


# Initialize the pre-trained classification model
obj_model = models.resnet50(pretrained=True)
obj_model.eval()

# CUDA if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
obj_model = obj_model.to(device)

imagenet_embed_word2vec = torch.load('/workspace/work/ral_revise_and_resubmit/ral_revise_resubmit/revised_feat_extraction/imagenet_word2vec.pt')
imagenet_embed_word2vec = imagenet_embed_word2vec.to(device)


from torchvision.transforms import functional as F
from torchvision import transforms

# Define the transform for preprocessing the RoI
imagenet_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


ml_elements['ROI_DET_MODEL'] = obj_model
ml_elements['ROI_PREPROCESS'] = imagenet_preprocess
ml_elements['imagenet_word2vec_embed'] = imagenet_embed_word2vec

hand_embedding = torch.load('/workspace/work/ral_revise_and_resubmit/ral_revise_resubmit/revised_feat_extraction/word2vec_hand_embedding.pt')
hand_embedding = torch.from_numpy(hand_embedding).to(device)

ml_elements['hand_embedding'] = hand_embedding

In [44]:
from PIL import Image, ImageSequence
import torch

def load_gif_frames(gif_path):
    """
    Load the frames of a GIF into a list.

    Args:
    - gif_path (str): Path to the GIF file.

    Returns:
    - List[Image.Image]: List of PIL Image objects representing the frames.
    """
    with Image.open(gif_path) as im:
        frames = [frame.copy().convert('RGB') for frame in ImageSequence.Iterator(im)]
    return frames



from glob import glob as glob

old_feat_files = '/workspace/data/data_folder/o2o/ral_features/full_features/*.pt'
file_list = glob(old_feat_files)

from tqdm import tqdm as tqdm
gif_folder = '/workspace/data/data_folder/o2o/gifs_11'


current_file = file_list[110]

current_dict = torch.load(current_file)

yt_id = current_dict['metadata']['yt_id']
frame_index = current_dict['metadata']['frame no.']

window_size = 5

# Loading the gif    
filename = yt_id + '_' + str(frame_index) + '_' + str(window_size) + '.gif'
import os
file_location = os.path.join(gif_folder, filename)

gif_frames = load_gif_frames(file_location)
print("DEBUG -1", gif_frames[5].mode)
# 3. Get the semantic features using zero shot methods
fname = '/workspace/work/misc/O2ONet/data/annotation.pkl'
import pickle as pkl

annotated_data = pkl.load( open(fname, 'rb'))

num_items = len(annotated_data)

for i in range(num_items):
    
    if annotated_data[i]['metadata']['yt_id'] == yt_id:
        if annotated_data[i]['metadata']['frame no.'] == frame_index:
            req_data = annotated_data[i]

hand_indices = []
object_dict = req_data['bboxes']
obj_keys = list(object_dict.keys())

for i, o in enumerate(obj_keys):
    temp_cat = object_dict[o]['class']
    if temp_cat == 'hand':
        hand_indices.append(i)


# 4. Get semantic embeddings of the bounding boxes
current_dict = get_semantic_embeddings(ml_elements, gif_frames, current_dict, hand_indices)

DEBUG -1 RGB
F 0 [[950.0, 317.0, 1221.0, 473.0], [705.0, 310.0, 946.0, 490.0], [6.0, 294.0, 272.0, 539.0], [406.0, 3.0, 428.0, 473.0]]


TypeError: zeros() received an invalid combination of arguments - got (tuple, dtype=torch.device), but expected one of:
 * (tuple of ints size, *, tuple of names names, torch.dtype dtype, torch.layout layout, torch.device device, bool pin_memory, bool requires_grad)
 * (tuple of ints size, *, Tensor out, torch.dtype dtype, torch.layout layout, torch.device device, bool pin_memory, bool requires_grad)
